In [1]:
import pandas as pd 
import caliperpy
import os
import matplotlib.pyplot as plt
import numpy as np

def read_mtx(filename, dk):
    # Reads a matrix
    # dk = caliperpy.TransCAD.connect()
    out_dict = {}
    try:
        mtx = dk.OpenMatrix(filename, "True")
        cores = dk.GetMatrixCoreNames(mtx)
        for c in cores: # type: ignore
            m1 = dk.CreateMatrixCurrency(mtx, c, None, None, None)
            out_dict[c] = np.nan_to_num(np.array(dk.GetMatrixValues(m1, None, None), dtype=np.float32))
    except Exception as e:
        print(e) # type: ignore
    # finally:
        # dk.Close()
        # caliperpy.TransCAD.disconnect()
    return out_dict

def read_length_mtx(filename, dk):
    # Reads a matrix
    # dk = caliperpy.TransCAD.connect()
    out_dict = {}
    try:
        mtx = dk.OpenMatrix(filename, "True")
        # cores = dk.GetMatrixCoreNames(mtx)
        # for c in cores: # type: ignore
        #     m1 = dk.CreateMatrixCurrency(mtx, c, None, None, None)
        #     out_dict[c] = np.nan_to_num(np.array(dk.GetMatrixValues(m1, None, None), dtype=np.float32))
        m1 = dk.CreateMatrixCurrency(mtx, "Length (Skim)", None, None, None)    
        out = np.nan_to_num(np.array(dk.GetMatrixValues(m1, None, None), dtype=np.float32))
    except Exception as e:
        print(e) # type: ignore
    # finally:
        # dk.Close()
        # caliperpy.TransCAD.disconnect()
    return out

SCENARIO_BASE_FOLDER = r'C:\models\Reno_TDM\scenarios'
SCENARIO_CONTROL = 'base_2022'
SCENARIO_CHANGE = 'base_2022_resnets_dl_s'
PERIODS = ['AM', 'MD', 'PM', 'NT']
PURPOSES = ['W_HBW', 'W_HBO', 'N_HBSCH', 'N_HBSHP', 'N_HBSR', 'N_HBO']

MODES = ['mc_sov', 'mc_hov2', 'mc_hov3', 'mc_auto_pay', 'mc_w_lb', 'mc_w_eb', 'mc_pnr_lb', 'mc_pnr_eb', 'mc_knr_lb', 'mc_knr_eb', 'mc_shuttle']

purp_list = {
    'W_HBW': ['v0', 'ilvi', 'ilvs', 'ihvi', 'ihvs'],
    'W_HBO': ['v0', 'vi', 'vs'],
    'N_HBSCH': ['v0', 'vi', 'vs'],
    'N_HBSHP': ['v0', 'vi', 'vs'],
    'N_HBSR': ['v0', 'vi', 'vs'],
    'N_HBO': ['v0', 'vi', 'vs']
}

dk = caliperpy.TransCAD.connect()

base_totals = {}
chng_totals = {}
skim = {}

for purp in purp_list.keys():
    print(f"Working on purpose {purp}")
    for per in PERIODS:
        input_base = read_mtx(os.path.join(SCENARIO_BASE_FOLDER, SCENARIO_CONTROL, 'output', 'resident', 'trip_matrices', f'pa_per_trips_{purp}_{per}.mtx'), dk)
        input_chng = read_mtx(os.path.join(SCENARIO_BASE_FOLDER, SCENARIO_CHANGE, 'output', 'resident', 'trip_matrices', f'pa_per_trips_{purp}_{per}.mtx'), dk)
        for mode in MODES:
            for vs in purp_list[purp]:
                if not (purp, per, mode) in base_totals.keys():
                    if mode in input_base.keys():
                        base_totals[(purp, per, mode)] = input_base[mode].sum()
                else:
                    if mode in input_base.keys():
                        base_totals[(purp, per, mode)] += input_base[mode].sum()

                if not (purp, per, mode) in chng_totals.keys():
                    if mode in input_chng.keys():
                        chng_totals[(purp, per, mode)] = input_chng[mode].sum()
                else:
                    if mode in input_chng.keys():
                        chng_totals[(purp, per, mode)] += input_chng[mode].sum()

            # if per in ['AM', 'PM']:
            #     skim[(purp, per)] = read_length_mtx(os.path.join(SCENARIO_BASE_FOLDER, SCENARIO_CONTROL, 'output', 'skims', 'roadway', f'avg_skim_{per}_{purp[:4]}_sov.mtx'), dk)
            # else:
            #     skim[(purp, per)] = read_length_mtx(os.path.join(SCENARIO_BASE_FOLDER, SCENARIO_CONTROL, 'output', 'skims', 'roadway', f'avg_skim_{per}_All_All_sov.mtx'), dk)
         

caliperpy.TransCAD.disconnect()

Connecting to TransCAD...
Working on purpose W_HBW
Working on purpose W_HBO
Working on purpose N_HBSCH
Working on purpose N_HBSHP
Working on purpose N_HBSR
Working on purpose N_HBO
Disconnected from TransCAD!


True

In [2]:
table = []
for k in base_totals.keys():
    table.append({'purpose': k[0], 'period': k[1], 'mode': k[2], 'base': base_totals[k], 'change': chng_totals[k]})

pd.DataFrame(table)

,purpose,period,mode,base,change
0,W_HBW,AM,mc_sov,280347.375000,280227.343750
1,W_HBW,AM,mc_hov2,36861.742188,36813.371094
2,W_HBW,AM,mc_hov3,4014.317871,4006.466797
3,W_HBW,AM,mc_auto_pay,33913.984375,34082.847656
4,W_HBW,AM,mc_w_lb,580.834961,593.859924
...,...,...,...,...,...
215,N_HBO,NT,mc_w_eb,150.257355,128.060455
216,N_HBO,NT,mc_pnr_lb,7.554767,6.440918
217,N_HBO,NT,mc_pnr_eb,7.532029,6.548762
218,N_HBO,NT,mc_knr_lb,377.541016,328.524902


In [3]:
pd.DataFrame(table).to_csv(r"C:\models\Reno_TDM\scenarios\base_2022_with_resnets\output\_summaries\Resnet_compare\resnet_mode_compare_dls.xlsx")